In [1]:
from itertools import pairwise
import ray
import torch
import numpy as np

In [2]:
# Interesting helper functions

GPU_FRACTION=0.2

# @ray.remote(num_gpus=GPU_FRACTION)
def rand_matmul(n, r, device):
    # print(f"rand_mat: {n=}, {m=}, {r=}")
    A = torch.randn(n, r).to(device)
    B = torch.randn(n, r).to(device)
    # print(f"rand_matmul: [{list(A.shape)}] @ [{list(B.shape)=}]")
    return torch.einsum("ar,br->ab", A, B)

# @ray.remote(num_gpus=GPU_FRACTION*4)
def bridge_tensor(a, c, device):
    """returns a matrix b of the smallest shape so that: a @ b @ c is valid"""
    # print(f"bridge_tensor: {a=}, {c=}")
    return torch.randn(a.shape[-1], c.shape[0]).to(device)

@ray.remote(num_gpus=GPU_FRACTION)
def mul_chain(n=10_000, r=1_000, l=2) -> torch.Tensor | None:
    # print(f"mul_chain: {n=}, {r=}, {l=}")
    device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

    res = torch.ones(n, n).to(device)
    for _ in range(l):
        a, c = [rand_matmul(n=n, r=r, device=device) for _ in range(2)]
        b = bridge_tensor(a, c, device=device)
        for _ in range(2**8): # This inner loop produces high arithmetic intensity
            res += a @ b @ c @ res
        # print(f"mul_chain: {list(d.shape)} = {list(a.shape)} @ {list(b.shape)} @ {list(c.shape)}")
    return res

In [3]:
# Make the GPUs sweat
NUM_GPU = 4
num_tasks = min(int(NUM_GPU / GPU_FRACTION) * 2, 1_000)
print(f"{num_tasks=}")
mc_ftrs = [mul_chain.remote(n=2**10, r=2**10, l=64) for _ in range(num_tasks)]
mcs = ray.get(mc_ftrs)
print([mc.shape for mc in mcs])

2026-05-21 21:34:12,620	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: 10.0.1.14:6379...
2026-05-21 21:34:12,650	INFO worker.py:2003 -- Connected to Ray cluster. View the dashboard at http://127.0.0.1:8265 


num_tasks=40


/home/nico/miniconda3/envs/drp/lib/python3.14/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


[torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size([1024, 1024]), torch.Size(

In [ ]:
# (0, 1, 2)
a = torch.randn(2, 3, 4)
print(a)

In [ ]:
a[0, 0, 2]

In [ ]:
ap = a.permute(2, 0, 1)
print(ap.shape)
print(ap)

In [ ]:
futures = [matmul_gpu.remote() for i in range(1000)]

results = ray.get(futures)
for r in results:
    print(r)

In [ ]:
# from ray.train import ScalingConfig
# from ray.train.torch import train_loop_utils, TorchTrainer
# from torchvision.datasets import CIFAR10
# from torchvision.transforms import ToTensor

# scaling_config = ScalingConfig(
#     num_workers=2,
#     use_gpu=True
# )

In [ ]:
# # 1. Download/load the dataset using torchvision
# torch_dataset = CIFAR10(root="/home/nico/ray-test/notebooks/data", train=True, download=True, transform=ToTensor())

# # 2. Convert to Ray Data
# ray_dataset = ray.data.from_torch(torch_dataset)

# print(ray_dataset.count())